# 01 Attention Scaling Factor Variance Verification

This notebook empirically validates the core variance identity underpinning scaled dot-product attention.

**Theory recap:**
For query and key vectors $q, k \in \mathbb{R}^{d_k}$ with components drawn $i.i.d.$ from $\mathcal{N}(0,1)$, the raw dot-product $S = q^\top k$ has:

$$\text{Var}(S) = d_k$$

Applying the $1/\sqrt{d_k}$ scaling correction restores unit variance:

$$\text{Var}\!\left(\frac{S}{\sqrt{d_k}}\right) = 1$$

**Structure:**
- **Part 1**: Isolate a single dimension ($d_k = 64$) and directly observe both variances.
- **Part 2**: Sweep across $d_k \in \{64, 128, 256, 512, 1024\}$, assert against theory, and print a summary table.

In [1]:
import torch
from tabulate import tabulate

torch.manual_seed(42)

---

## Part 1: Single Dimension Proof of Concept

Before sweeping across dimensions, we isolate one case: $d_k = 64$.

We run 20 independent trials. Each trial draws $100{,}000$ fresh $(q, k)$ pairs (one scalar dot-product $S = q^\top k$ per pair) and computes the empirical variance. We then report the **mean ± std** across trials, a stable estimate that also shows how tight the distribution of variance estimates is around the theoretical target.

**Expected result:**
- Unscaled variance $\approx 64 \pm \epsilon$ (mean converges to $d_k$, std should be small)
- Scaled variance $\approx 1.0 \pm \epsilon$

In [2]:
d_k        = 64
n_samples  = 100_000
num_trials = 20

vars_unscaled, vars_scaled = [], []

for _ in range(num_trials):
    Q = torch.randn(n_samples, d_k)
    K = torch.randn(n_samples, d_k)
    S_unscaled = (Q * K).sum(dim=-1)    # shape: (n_samples,), one scalar per (q, k) pair
    S_scaled   = S_unscaled / (d_k ** 0.5)
    vars_unscaled.append(torch.var(S_unscaled).item())
    vars_scaled.append(torch.var(S_scaled).item())

t_u = torch.tensor(vars_unscaled)
t_s = torch.tensor(vars_scaled)

print(f"d_k = {d_k}  ({num_trials} trials, {n_samples:,} samples each)")
print(f"  Unscaled: mean: {t_u.mean():.4f}  std: {t_u.std():.4f}   (theory: {float(d_k):.1f})")
print(f"  Scaled:   mean: {t_s.mean():.6f}  std: {t_s.std():.6f}  (theory: 1.0)")

d_k = 64  (20 trials, 100,000 samples each)
  Unscaled: mean: 64.0723  std: 0.2890   (theory: 64.0)
  Scaled:   mean: 1.001129  std: 0.004515  (theory: 1.0)


---

## Part 2: Sweep Across Standard Hidden Dimensions

The single-dim result holds, but the claim is general: variance scales as $d_k$ for any dimension.

We repeat the experiment across five key dimensions. Each row cites published 2026 technical reports where available; higher values are included to empirically confirm the scaling law holds at every scale.

| $d_k$ | Representative Models |
|------:|-----------------------|
| 64 | Legacy tier: BERT-Base (768d, 12h, 2018), GPT-2 Small (768d, 12h, 2019). No major 2026 frontier model uses $d_k = 64$; the field has converged on 128 as the new floor. |
| 128 | **Nemotron 3 Ultra 550B** (8192d, 64h, GQA, Jun 2026 †), **Arcee Trinity Large 398B** (3072d, 48h, GQA, Apr 2026 ‡), **K-EXAONE 236B** (64h, GQA/SWA, Jan 2026 §) |
| 192 | **Kimi K2.5** (7168d, 64h, MLA, 2026 ¶), a non-power-of-2 head dimension arising from MLA's decoupled RoPE design: $d_k = d_k^{\text{nope}} + d_k^{\text{rope}} = 128 + 64 = 192$ |
| 256 | **Qwen3-8B** (8192d, 32h, GQA, May 2025 ∥); among the few confirmed GQA models with $d_k = 256$. |
| 512 | Theoretical sweep point: no confirmed 2026 production model uses $d_k = 512$. Included to empirically validate $\text{Var}(S) = d_k$ at scale. |
| 1024 | Theoretical sweep point: frontier models have converged on $d_k \in \{128, 192, 256\}$. Included for completeness of the linear scaling proof. |

*† arXiv:2606.15007, Nemotron 3 Ultra, NVIDIA, Jun 2026.*
*‡ arXiv:2602.17004, Arcee Trinity Large Technical Report, Apr 2026.*
*§ arXiv:2601.01739, K-EXAONE Technical Report, LG AI Research, Jan 2026.*
*¶ glennklockwood.com/garden/Kimi-K2.5, config: `qk_nope_head_dim=128`, `qk_rope_head_dim=64`.*
*∥ arXiv:2505.09388, Qwen3 Technical Report, Alibaba, May 2025.*

In [3]:
DIMENSIONS = [64, 128, 192, 256, 512, 1024]
N_SAMPLES  = 100_000
NUM_TRIALS = 20

results = []

for d_k in DIMENSIONS:
    vars_unscaled, vars_scaled = [], []

    for _ in range(NUM_TRIALS):
        Q = torch.randn(N_SAMPLES, d_k)
        K = torch.randn(N_SAMPLES, d_k)
        S_unscaled = (Q * K).sum(dim=-1)    # shape: (N_SAMPLES,)
        S_scaled   = S_unscaled / (d_k ** 0.5)
        vars_unscaled.append(torch.var(S_unscaled).item())
        vars_scaled.append(torch.var(S_scaled).item())

    t_u = torch.tensor(vars_unscaled)
    t_s = torch.tensor(vars_scaled)

    results.append((
        d_k,
        float(d_k), t_u.mean().item(), t_u.std().item(),
        1.0,         t_s.mean().item(), t_s.std().item(),
    ))

In [4]:
TOLERANCE = 0.05

for d_k, target_u, mean_u, std_u, target_s, mean_s, std_s in results:
    assert abs(mean_u - target_u) < TOLERANCE * target_u, (
        f"d_k={d_k}: unscaled mean var {mean_u:.4f} deviates >5% from target {target_u}"
    )
    assert abs(mean_s - target_s) < TOLERANCE * target_s, (
        f"d_k={d_k}: scaled mean var {mean_s:.6f} deviates >5% from target {target_s}"
    )

print("All assertions passed: trial-averaged results match theoretical targets within 5% tolerance.")

All assertions passed: trial-averaged results match theoretical targets within 5% tolerance.


In [5]:
headers = [
    "d_k",
    "Target Unscaled",
    "Unscaled Mean ± Std",
    "Target Scaled",
    "Scaled Mean ± Std",
]

rows = [
    [d_k, f"{tu:.1f}", f"{mu:.4f} ± {su:.4f}", f"{ts:.1f}", f"{ms:.6f} ± {ss:.6f}"]
    for d_k, tu, mu, su, ts, ms, ss in results
]

print(tabulate(rows, headers=headers, tablefmt="github"))
print(f"\n({NUM_TRIALS} trials × {N_SAMPLES:,} samples per trial)")

|   d_k |   Target Unscaled | Unscaled Mean ± Std   |   Target Scaled | Scaled Mean ± Std   |
|-------|-------------------|-----------------------|-----------------|---------------------|
|    64 |                64 | 64.0362 ± 0.2429      |               1 | 1.000565 ± 0.003795 |
|   128 |               128 | 128.2019 ± 0.4032     |               1 | 1.001578 ± 0.003150 |
|   192 |               192 | 192.1526 ± 0.9224     |               1 | 1.000795 ± 0.004804 |
|   256 |               256 | 255.8606 ± 1.2455     |               1 | 0.999455 ± 0.004865 |
|   512 |               512 | 512.0762 ± 2.1014     |               1 | 1.000149 ± 0.004104 |
|  1024 |              1024 | 1024.4449 ± 3.9615    |               1 | 1.000435 ± 0.003869 |

(20 trials × 100,000 samples per trial)


---

## Part 3: Softmax Saturation

The variance proof established *why* unscaled logits are dangerous. This section makes the consequence concrete.

We feed the attention logits into softmax and measure two quantities for each $d_k$:

- **Max probability**: weight assigned to the single most-attended token. Uniform distribution gives $1 / \text{seq\_len}$; saturated gives $\approx 1.0$.
- **Entropy** $H = -\sum_i \sigma_i \log \sigma_i$: how spread the attention is. Uniform gives $\log(\text{seq\_len})$; a one-hot delta gives $0$.

**Prediction:** as $d_k$ grows, unscaled logits push softmax toward a one-hot: max probability → 1, entropy → 0. Scaled logits stay near uniform regardless of $d_k$.

In [6]:
SEQ_LEN       = 50     # number of tokens in one attention row
N_SOFTMAX     = 1_000  # sequences to average over, for more stable estimates

uniform_entropy = torch.log(torch.tensor(float(SEQ_LEN))).item()
softmax_results = []

for d_k in DIMENSIONS:
    max_probs_u, max_probs_s = [], []
    entropies_u, entropies_s = [], []

    for _ in range(N_SOFTMAX):
        logits_u = torch.randn(SEQ_LEN) * (d_k ** 0.5)   # Var = d_k
        logits_s = logits_u / (d_k ** 0.5)               # Var = 1

        attn_u = torch.softmax(logits_u, dim=0)
        attn_s = torch.softmax(logits_s, dim=0)

        max_probs_u.append(attn_u.max().item())
        max_probs_s.append(attn_s.max().item())
        entropies_u.append(-(attn_u * attn_u.clamp(min=1e-9).log()).sum().item())
        entropies_s.append(-(attn_s * attn_s.clamp(min=1e-9).log()).sum().item())

    softmax_results.append([
        d_k,
        f"{sum(max_probs_u) / N_SOFTMAX:.4f}",
        f"{sum(max_probs_s) / N_SOFTMAX:.4f}",
        f"{sum(entropies_u) / N_SOFTMAX:.4f}",
        f"{sum(entropies_s) / N_SOFTMAX:.4f}",
        f"{uniform_entropy:.4f}",
    ])

headers = ["d_k", "Unscaled Max P", "Scaled Max P", "Unscaled Entropy", "Scaled Entropy", "Uniform Entropy"]
print(tabulate(softmax_results, headers=headers, tablefmt="github"))
print(f"\n(seq_len={SEQ_LEN}, averaged over {N_SOFTMAX:,} sequences)")

|   d_k |   Unscaled Max P |   Scaled Max P |   Unscaled Entropy |   Scaled Entropy |   Uniform Entropy |
|-------|------------------|----------------|--------------------|------------------|-------------------|
|    64 |           0.8032 |         0.1262 |             0.5435 |           3.4445 |             3.912 |
|   128 |           0.8698 |         0.1272 |             0.3448 |           3.4401 |             3.912 |
|   192 |           0.8795 |         0.1231 |             0.3044 |           3.4478 |             3.912 |
|   256 |           0.8966 |         0.1229 |             0.264  |           3.4487 |             3.912 |
|   512 |           0.9273 |         0.1249 |             0.1801 |           3.4431 |             3.912 |
|  1024 |           0.9541 |         0.1254 |             0.1134 |           3.4461 |             3.912 |

(seq_len=50, averaged over 1,000 sequences)


---

## Part 4: Gradient Capacity via the Jacobian Trace

During backpropagation, gradients flow to the logits through the softmax Jacobian $J = \text{diag}(\sigma) - \sigma \sigma^\top$. The on-diagonal entry for each token is:

$$J_{ii} = \sigma_i(1 - \sigma_i)$$

This is exactly the sigmoid saturation function. It is zero when a token is fully attended ($\sigma_i \approx 1$) and zero when a token is ignored ($\sigma_i \approx 0$). Gradient only flows through tokens with intermediate attention weight.

The **trace** of $J$ sums this across all tokens:

$$\text{tr}(J) = \sum_{i=1}^{n} \sigma_i(1 - \sigma_i) = 1 - \|\sigma\|_2^2$$

This is the total gradient capacity of the layer. When attention collapses to a one-hot, every token ends up with either $\sigma_i \approx 1$ or $\sigma_i \approx 0$, making every diagonal entry — and therefore $\text{tr}(J)$ — approach zero. No logit receives a useful gradient, regardless of how large the upstream loss is.

$$\text{tr}(J) \approx 1 \; \text{(uniform, gradient healthy)} \qquad \text{tr}(J) \to 0 \; \text{(one-hot, gradient dead)}$$

We also report the **effective attention span** $= 1/\|\sigma\|_2^2$, the participation ratio of the distribution. It directly answers: how many tokens is the model actually attending to? It collapses from hundreds of tokens toward 1 as saturation increases.

We use $\text{seq\_len} = 512$ (BERT's context length) to match production-scale attention rows.

In [7]:
JAC_SEQ_LEN = 512   # BERT context length
N_JAC       = 1_000  # sequences to average over
trace_results = []

for d_k in DIMENSIONS:
    traces_u, traces_s = [], []
    spans_u,  spans_s  = [], []

    for _ in range(N_JAC):
        logits_u = torch.randn(JAC_SEQ_LEN) * (d_k ** 0.5)
        logits_s = logits_u / (d_k ** 0.5)

        sigma_u = torch.softmax(logits_u, dim=0)
        sigma_s = torch.softmax(logits_s, dim=0)

        # tr(J) = sum_i sigma_i(1 - sigma_i) = 1 - ||sigma||^2
        traces_u.append((sigma_u * (1 - sigma_u)).sum().item())
        traces_s.append((sigma_s * (1 - sigma_s)).sum().item())

        # effective attention span = 1 / ||sigma||^2  (participation ratio)
        spans_u.append((1.0 / (sigma_u ** 2).sum()).item())
        spans_s.append((1.0 / (sigma_s ** 2).sum()).item())

    trace_results.append([
        d_k,
        f"{sum(traces_u) / N_JAC:.4f}",
        f"{sum(traces_s) / N_JAC:.4f}",
        f"{sum(spans_u)  / N_JAC:.1f}",
        f"{sum(spans_s)  / N_JAC:.1f}",
    ])

headers = ["d_k", "tr(J) unscaled  (→ 0)", "tr(J) scaled  (~1.0)",
           "Span unscaled  (→ 1)", f"Span scaled  (/{JAC_SEQ_LEN})"]
print(tabulate(trace_results, headers=headers, tablefmt="github"))
print(f"\n(seq_len={JAC_SEQ_LEN}, averaged over {N_JAC:,} sequences)")
print("\ntr(J) → 0: attention layer can no longer pass gradients to W_Q and W_K.")
print("Span → 1: a single token captures all attention; the weight matrices are frozen.")

|   d_k |   tr(J) unscaled  (→ 0) |   tr(J) scaled  (~1.0) |   Span unscaled  (→ 1) |   Span scaled  (/512) |
|-------|-------------------------|------------------------|------------------------|-----------------------|
|    64 |                  0.3739 |                 0.9947 |                    2   |                 196.5 |
|   128 |                  0.2525 |                 0.9948 |                    1.5 |                 196.8 |
|   192 |                  0.2257 |                 0.9948 |                    1.5 |                 199   |
|   256 |                  0.1903 |                 0.9947 |                    1.4 |                 196   |
|   512 |                  0.1372 |                 0.9948 |                    1.3 |                 197.1 |
|  1024 |                  0.0967 |                 0.9948 |                    1.2 |                 197.1 |

(seq_len=512, averaged over 1,000 sequences)

tr(J) → 0: attention layer can no longer pass gradients to W_Q and W_K.
S